# 7.12 模型會回答新組合嗎？


訓練見紅圓、藍圓、藍方，留下紅方。紅與方都出現過，新的只是它們同時出現的組合。這種切法能問模型會不會組合已有屬性，和考從未見過的顏色不同。

先把任務寫清楚：給一個物件的顏色與形狀，問「是什麼形狀？」。例如「紅色方塊是什麼形狀？」的答案是「方塊」。這份小材料有兩顏色×兩形狀，共四種組合；答案由形狀決定。

| 屬性組合 | 問題 | 答案 | 用途 |
| --- | --- | --- | --- |
| 紅、圓 | 紅色圓形是什麼形狀？ | 圓形 | 訓練 |
| 藍、圓 | 藍色圓形是什麼形狀？ | 圓形 | 訓練 |
| 藍、方 | 藍色方塊是什麼形狀？ | 方塊 | 訓練 |
| 紅、方 | 紅色方塊是什麼形狀？ | 方塊 | 留出檢查 |

下面只安排四格的切分，先檢查訓練與留出沒有交集；沒有訓練模型或產生答題成績。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
all_pairs = [(color, shape) for color in ["紅", "藍"] for shape in ["圓", "方"]]
held_out = {("紅", "方")}
train = [pair for pair in all_pairs if pair not in held_out]
print("訓練組合", train)
print("留出組合", held_out)
assert not (set(train) & held_out)

輸入屬性列表，第一行是兩層清單迴圈的簡寫：每種顏色都配每種形狀，因此先紅圓、紅方，再藍圓、藍方。`held_out` 用集合保存留出項，`if pair not in held_out` 只保留其餘。輸出訓練三組，檢查兩組集合沒有交集；這個檢查只驗證分組規則。

同一屬性組合的所有措辭版本要一起分組，不能只分開表面文字。例如「紅色方塊是什麼形狀？」和「一個紅方形，請說出形狀」都屬於紅方，答案都是方塊；兩題都應留出，不能讓一題進訓練、另一題進留出後稱新組合。資料工具應記錄顏色與形狀欄位，按這些欄位分組。每個組合連同它的不同問法，可以稱為一個題目家族。

練習把held_out改成 `{("藍","圓"),("藍","方")}`，先預測訓練只兩種紅組合、訓練中缺藍，再執行核對。寫一句「這次測未見顏色」說明目標變化。恢復紅方留出後，確認紅、藍、圓、方仍各出現至少一次，才回到新組合的原問題。

<details>
<summary>補充：獨立實驗與閱讀範圍</summary>

專案另有12種屬性組合的獨立實驗；配方見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，逐題紀錄見[完整實測報告](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/sft.json)。那份比較的支線更新預算不同，且只有一個種子和少量組合，不能當作通則。讀這個四格示例不需要運行該實驗；本節只示範材料與切分，沒有新訓練結果。

</details>
